# Realtime voice model — RVC v2 fine-tune (full cloud)

Trains a **real-time voice-conversion** model on Kaggle, in the same all-cloud
style as the F5-TTS notebook. Different problem on purpose:

- **F5-TTS / XTTS** = text -> speech (TTS). Can't run a live call through them.
- **RVC v2** = speech -> speech (voice conversion): keep the words/prosody from
  a microphone, swap only the voice identity. This is what powers real-time
  voice changers, and the trained `.pth` + `.index` load straight into
  `w-okada/voice-changer` for live use.

We **reuse the dataset you already built** (`cyttic/lecturer-ru-dataset`) — RVC
needs no transcripts, just clean single-speaker audio, which is exactly what the
F5 pipeline already produced.

**Before running:** Settings -> Accelerator -> **GPU**; Add-ons -> Secrets ->
`HF_TOKEN` (write token). Then edit CONFIG and Run All.

## 0 · Install RVC + pretrained assets

In [ ]:
RVC = '/kaggle/working/Retrieval-based-Voice-Conversion-WebUI'
import os, shutil
# torch>=2.6 defaults torch.load to weights_only=True, which rejects fairseq's
# hubert checkpoint and the RVC pretrained G/D. Flip it back for the whole run;
# !python subprocesses inherit this env var.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

!git clone -q https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC}
os.chdir(RVC)

# Keep Kaggle's CUDA torch (don't let requirements reinstall it).
!sed -i -E '/^(fairseq|torch|torchaudio|torchvision)([=<>! ]|$)/d' requirements.txt
# requirements.txt aborts as a whole if any one package fails metadata generation
# (leaving everything uninstalled), so install RVC's runtime deps explicitly too.
# `av` (PyAV) + `ffmpeg-python` are both used by infer/lib/audio.py.
!pip -q install -r requirements.txt
!pip -q install av ffmpeg-python praat-parselmouth pyworld torchcrepe faiss-cpu \
    tensorboardX tensorboard matplotlib datasets soundfile huggingface_hub
# fairseq 0.12 has no Python-3.12 wheel, and its setup.py imports torch/cython at
# metadata time -> pip's build isolation hides them -> "egg_info failed". Install
# the 3.12-friendly fork WITHOUT build isolation so it sees Kaggle's torch/cython.
!pip -q install --no-build-isolation cython omegaconf
!pip -q install --no-build-isolation "git+https://github.com/One-sixth/fairseq.git"

from huggingface_hub import hf_hub_download
def grab(fn, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy(hf_hub_download('lj1995/VoiceConversionWebUI', fn), dest)
grab('hubert_base.pt',                'assets/hubert/hubert_base.pt')
grab('rmvpe.pt',                      'assets/rmvpe/rmvpe.pt')
grab('pretrained_v2/f0G40k.pth',      'assets/pretrained_v2/f0G40k.pth')
grab('pretrained_v2/f0D40k.pth',      'assets/pretrained_v2/f0D40k.pth')
print('RVC ready at', RVC)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ============================== CONFIG ==============================
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')   # write token
os.environ['HF_TOKEN'] = HF_TOKEN

HF_USER         = 'cyttic'
HF_DATASET_REPO = f'{HF_USER}/lecturer-ru-dataset'   # reuse the F5 clips (raw audio)
HF_MODEL_REPO   = f'{HF_USER}/lecturer-ru-rvc'       # trained model goes here

NAME       = 'lecturer_ru'   # experiment + output model name
SR, SR_HZ  = '40k', 40000    # 40k v2 = the realtime sweet spot (low latency, good quality)
EPOCHS     = 150             # RVC epochs are cheap; 100-300 typical for ~30 min audio
BATCH      = 8               # lower to 4-6 if the T4 OOMs
SAVE_EVERY = 50
# ===================================================================

## 1 · Pull the dataset from HF -> a wav folder

RVC's preprocessor wants a folder of wavs. No transcripts needed.

In [ ]:
import io, soundfile as sf
from datasets import load_dataset, Audio

WAVS = '/kaggle/working/wavs'
os.makedirs(WAVS, exist_ok=True)
ds = load_dataset(HF_DATASET_REPO, split='train', token=HF_TOKEN)
ds = ds.cast_column('audio', Audio(decode=False))   # raw bytes, no decoder
for i, r in enumerate(ds):
    d, s = sf.read(io.BytesIO(r['audio']['bytes']), dtype='float32')
    sf.write(f'{WAVS}/clip_{i:04d}.wav', d, s)
print(len(ds), 'wavs ->', WAVS)

## 2 · Preprocess (slice + resample to 40k)

In [ ]:
!python infer/modules/train/preprocess.py {WAVS} {SR_HZ} 2 logs/{NAME} False 3.0
print('preprocess output:', os.listdir(f'logs/{NAME}'))

## 3 · Extract pitch (RMVPE) + content features (ContentVec, v2/768-dim)

In [ ]:
# RMVPE pitch -> 2a_f0 / 2b-f0nsf ; ContentVec features -> 3_feature768
!python infer/modules/train/extract/extract_f0_rmvpe.py 1 0 0 logs/{NAME} True
!python infer/modules/train/extract_feature_print.py cuda:0 1 0 0 logs/{NAME} v2 True
print('f0:', len(os.listdir(f'logs/{NAME}/2a_f0')),
      '| features:', len(os.listdir(f'logs/{NAME}/3_feature768')))

## 4 · Build the training filelist + config

Replicates what the WebUI's "Train" button does headlessly.

In [ ]:
import random, glob
exp = f'logs/{NAME}'
gt, ft   = f'{exp}/0_gt_wavs', f'{exp}/3_feature768'
f0, f0n  = f'{exp}/2a_f0', f'{exp}/2b-f0nsf'

names = (set(x.rsplit('.', 1)[0] for x in os.listdir(gt))
         & set(x.rsplit('.', 1)[0] for x in os.listdir(ft))
         & set(x.split('.')[0] for x in os.listdir(f0))
         & set(x.split('.')[0] for x in os.listdir(f0n)))

A = os.path.abspath
opt = [f'{A(gt)}/{n}.wav|{A(ft)}/{n}.npy|{A(f0)}/{n}.wav.npy|{A(f0n)}/{n}.wav.npy|0'
       for n in names]

# a couple of "mute" reference rows stabilise training on small datasets
m = A('logs/mute')
opt += [f'{m}/0_gt_wavs/mute40k.wav|{m}/3_feature768/mute.npy|'
        f'{m}/2a_f0/mute.wav.npy|{m}/2b-f0nsf/mute.wav.npy|0'] * 2

random.shuffle(opt)
open(f'{exp}/filelist.txt', 'w').write('\n'.join(opt))

# config path moved across RVC versions -> locate the 40k v2 config, don't hardcode
cfg = next(c for c in glob.glob('configs/**/40k*.json', recursive=True) if 'v2' in c)
shutil.copy(cfg, f'{exp}/config.json')
print(len(opt), 'training items | config:', cfg)

## 5 · Train (from the v2 40k pretrained generator + discriminator)

In [ ]:
# matplotlib >= 3.8 removed FigureCanvasAgg.tostring_rgb(), which RVC's tensorboard
# spectrogram logging still calls -> patch it to buffer_rgba() before training.
_u = 'infer/lib/train/utils.py'
_s = open(_u).read()
_s = _s.replace('np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")',
                'np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)')
_s = _s.replace('data.reshape(fig.canvas.get_width_height()[::-1] + (3,))',
                'data.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:, :, :3]')
open(_u, 'w').write(_s)

# -l 1 keeps only the latest G/D checkpoint (disk-safe, like the F5 lesson).
# -sw 1 writes the small inference model to assets/weights/{NAME}.pth.
!python infer/modules/train/train.py -e {NAME} -sr {SR} -f0 1 -bs {BATCH} -g 0 \
  -te {EPOCHS} -se {SAVE_EVERY} \
  -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth \
  -l 1 -c 0 -sw 1 -v v2
# T4 OOM? lower BATCH in CONFIG. No sound later? check the speaker had enough clean audio.

## 6 · Build the retrieval index (faiss)

The `.index` is the "retrieval" half of RVC — it sharpens timbre at inference.

In [ ]:
import numpy as np, faiss, glob
feats = sorted(glob.glob(f'logs/{NAME}/3_feature768/*.npy'))
big = np.concatenate([np.load(f) for f in feats], axis=0)
print('feature matrix:', big.shape)

n_ivf = max(1, min(int(16 * np.sqrt(big.shape[0])), big.shape[0] // 39))
index = faiss.index_factory(768, f'IVF{n_ivf},Flat')
index.train(big)
for i in range(0, big.shape[0], 8192):
    index.add(big[i:i + 8192])
out = f'logs/{NAME}/added_{NAME}.index'
faiss.write_index(index, out)
print('wrote', out)

## 7 · Upload model + index to HuggingFace

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_MODEL_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(path_or_fileobj=f'assets/weights/{NAME}.pth',
                path_in_repo=f'{NAME}.pth', repo_id=HF_MODEL_REPO)
api.upload_file(path_or_fileobj=glob.glob(f'logs/{NAME}/added_*.index')[0],
                path_in_repo=f'{NAME}.index', repo_id=HF_MODEL_REPO)
print('done ->', f'https://huggingface.co/{HF_MODEL_REPO}')

## Using it in real time

Download the two files from your HF model repo:

- `lecturer_ru.pth` — the voice model
- `lecturer_ru.index` — the retrieval index

Then in [`w-okada/voice-changer`](https://github.com/w-okada/voice-changer):
1. Pick **RVC** as the engine, load the `.pth` + `.index`.
2. Set a virtual microphone as output (PipeWire/PulseAudio null-sink on Linux,
   VB-CABLE on Windows) and select it as the mic in your call app.
3. Tune **chunk size** (latency vs quality), **index ratio** (timbre strength),
   and **pitch shift** if your voice's range differs from the lecturer's.

40k v2 runs in real time on your RTX 2080 Super at ~100-200 ms.